In [5]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [6]:
import os

In [7]:
from scipy import io as sio
from scipy import linalg as la

In [4]:
from mayavi import mlab

KeyboardInterrupt: 

# Hermitian case

In [5]:
%matplotlib

Using matplotlib backend: QtAgg


In [8]:
GBZ_info = sio.loadmat("data/GBZ_info_1.mat")

In [8]:
print(GBZ_info)

{'__header__': b'MATLAB 5.0 MAT-file Platform: posix, Created on: Sat Sep  2 19:06:52 2023', '__version__': '1.0', '__globals__': [], 'model_params': array([[3.+0.j, 3.+0.j, 4.+0.j, 4.+0.j, 0.+1.j, 0.+1.j]]), 'phi1_list': array([[0.06159986, 0.12319971, 0.18479957, 0.24639942, 0.30799928,
        0.36959914, 0.43119899, 0.49279885, 0.5543987 , 0.61599856,
        0.67759842, 0.73919827, 0.80079813, 0.86239798, 0.92399784,
        0.9855977 , 1.04719755, 1.10879741, 1.17039726, 1.23199712,
        1.29359698, 1.35519683, 1.41679669, 1.47839654, 1.5399964 ,
        1.60159625, 1.66319611, 1.72479597, 1.78639582, 1.84799568,
        1.90959553, 1.97119539, 2.03279525, 2.0943951 , 2.15599496,
        2.21759481, 2.27919467, 2.34079453, 2.40239438, 2.46399424,
        2.52559409, 2.58719395, 2.64879381, 2.71039366, 2.77199352,
        2.83359337, 2.89519323, 2.95679309, 3.01839294, 3.0799928 ]]), 'phi2_list': array([[0.06159986, 0.12319971, 0.18479957, 0.24639942, 0.30799928,
        0.3695

In [9]:
ux1, ux2, uy1, uy2, vx1, vx2 = GBZ_info['model_params'].flatten()

In [6]:
print(vx1)
print(ux1)
print(uy1)

1j
(3+0j)
(4+0j)


In [10]:
phi1_list = GBZ_info['phi1_list'].flatten()
phi2_list = GBZ_info['phi2_list'].flatten()

In [11]:
def Hamiltonian_matrix(k1, k2, ux, uy, vx):
    H = np.array([
        [2 * uy * np.cos(k2), ux + vx * np.exp(1j*k1)],
        [ux + vx * np.exp(-1j*k1), 2 * vx * np.cos(k2)]
    ])
    return H

In [12]:
ux = ux1
uy = uy1
vx = vx1
# vx = -uy

In [13]:
def char_poly(E, beta1, beta2, ux, uy, vx):
    return (E**2 - (vx/beta2 + vx * beta2 + uy / beta2 + uy * beta2) * E +
            vx * uy * beta2**2 + uy * vx / (beta2**2) - ux * vx / beta1 - ux * vx * beta1 + vx * uy + vx * uy - ux **2 - vx **2) 

In [14]:
print(phi2_list/np.pi)

[0.01960784 0.03921569 0.05882353 0.07843137 0.09803922 0.11764706
 0.1372549  0.15686275 0.17647059 0.19607843 0.21568627 0.23529412
 0.25490196 0.2745098  0.29411765 0.31372549 0.33333333 0.35294118
 0.37254902 0.39215686 0.41176471 0.43137255 0.45098039 0.47058824
 0.49019608 0.50980392 0.52941176 0.54901961 0.56862745 0.58823529
 0.60784314 0.62745098 0.64705882 0.66666667 0.68627451 0.70588235
 0.7254902  0.74509804 0.76470588 0.78431373 0.80392157 0.82352941
 0.84313725 0.8627451  0.88235294 0.90196078 0.92156863 0.94117647
 0.96078431 0.98039216]


In [27]:
def test_Hermitian_y_major(phi1_index:int):
    global phi1_list, ux, uy, vx
    phi1 = phi1_list[phi1_index]
    print(phi1/np.pi)
    k1 = phi1/2
    k2 = np.linspace(0,2*np.pi, 50)
    E_list = np.zeros((4, len(k2)))
    for j in range(len(k2)):
        E_list[:2,j] = la.eigh(Hamiltonian_matrix(-k1, k2[j], ux, uy, vx))[0]
        E_list[2:,j] = la.eigh(Hamiltonian_matrix(np.pi - k1, k2[j], ux, uy, vx))[0]
        
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    ax.set_xlabel("Re$\\beta_1$")
    ax.set_ylabel("Im$\\beta_1$")
    ax.set_zlabel("E")
    print("Theoretical")
    for j in range(4):
        ax.plot(np.cos(k2), np.sin(k2), E_list[j,:])
    #     if(j < 2):
    #         print(np.max(np.abs(char_poly(E_list[j,:], np.exp(1j*k1), np.exp(1j*k2), ux, uy, vx))))
    #     else:
    #         print(np.max(np.abs(char_poly(E_list[j,:], np.exp(1j*k1), np.exp(1j*(np.pi-k2)), ux, uy, vx))))

    
    with open("data/GBZ_data_3/y_major_%d.pkl"%(phi1_index), "rb") as fp:
        GBZ_segments = pickle.load(fp)
    
    print("Numerical")
    for seg in GBZ_segments:
        # print(seg[0,2] - np.exp(-1j*k2))
        # print(np.log(seg[0,2]).imag/np.pi)
        # print(k2/np.pi)
        ax.plot(seg[:,2].real, seg[:,2].imag, seg[:,0].real, '.')
        # print(np.max(np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2], ux, uy, vx))))
        # print(np.max(np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2]*np.exp(1j*phi2), ux, uy, vx))))
    
    plt.show()

In [21]:
def test_Hermitian(phi2_index:int):
    global phi2_list, ux, uy, vx
    phi2 = phi2_list[phi2_index]
    print(phi2/np.pi)
    k2 = phi2/2
    k1 = np.linspace(0,2*np.pi, 50)
    E_list = np.zeros((4, len(k1)))
    for j in range(len(k1)):
        E_list[:2,j] = la.eigh(Hamiltonian_matrix(k1[j], -k2, ux, uy, vx))[0]
        E_list[2:,j] = la.eigh(Hamiltonian_matrix(k1[j], np.pi-k2, ux, uy, vx))[0]
        
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    ax.set_xlabel("Re$\\beta_1$")
    ax.set_ylabel("Im$\\beta_1$")
    ax.set_zlabel("E")
    ax = fig.gca()
    # print("Theoretical")
    for j in range(4):
        ax.plot(np.cos(k1), np.sin(k1), E_list[j,:])
    #     if(j < 2):
    #         print(np.max(np.abs(char_poly(E_list[j,:], np.exp(1j*k1), np.exp(1j*k2), ux, uy, vx))))
    #     else:
    #         print(np.max(np.abs(char_poly(E_list[j,:], np.exp(1j*k1), np.exp(1j*(np.pi-k2)), ux, uy, vx))))

    
    with open("data/GBZ_data_3/x_major_%d.pkl"%(phi2_index), "rb") as fp:
        GBZ_segments = pickle.load(fp)
    
    print(len(GBZ_segments))
    print("Numerical")
    for j, seg in enumerate(GBZ_segments):
        # print(seg[0,2] - np.exp(-1j*k2))
        # print(np.log(seg[0,2]).imag/np.pi)
        # print(k2/np.pi)
        # if(seg.shape[0] > 47):
        ax.plot(seg[:,1].real, seg[:,1].imag, seg[:,0].real, '--')
        # ax.plot(seg[:,0].real, seg[:,0].imag)
        # print(np.log(seg[:,2]).imag/np.pi)
        # print(np.log(seg[:,1]).imag/np.pi)
        # print(np.max(np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2], ux, uy, vx))))
        # print(np.max(np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2]*np.exp(1j*phi2), ux, uy, vx))))
        print(j)
        print(seg[:,0])
        print((np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2], ux, uy, vx))))
        print((np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2]*np.exp(1j*phi2), ux, uy, vx))))
        print((np.abs(char_poly(seg[:,0],seg[:,1], seg[:,2]*np.exp(-1j*phi2), ux, uy, vx))))
        # plt.pause(1)
        # input()
    
    plt.show()

In [17]:
def plot_BZ_band(ax):
    global vx, ux, uy
    k1 = np.linspace(-np.pi, np.pi, 100)
    k2 = np.linspace(-np.pi, np.pi, 100)
    K1, K2 = np.meshgrid(k1, k2)
    E = np.zeros((2,100,100))
    for i in range((100)):
        for j in range((100)):
            E[:, i, j] = la.eigh(Hamiltonian_matrix(K1[i,j], K2[i,j], ux, uy, vx))[0]
    ax.plot_surface(K1, K2, E[0,:,:])
    ax.plot_surface(K1, K2, E[1,:,:])

In [18]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
plot_BZ_band(ax)
plt.show()

In [30]:
def sweep_x_major():
    global ux, uy, vx
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    for j in range(20, 25):
        if(os.path.exists("data/GBZ_data_3/x_major_%d.pkl"%(j))):
            test_BZ_x_major(j, ax)
    plt.show()

In [32]:
def test_BZ_x_major(phi2_index:int, ax):
    global ux, uy, vx
    # fig = plt.figure()
    # ax = fig.add_subplot(projection='3d')
    # plot_BZ_band(ax)
    max_diff = 0.0
    with open("data/GBZ_data_3/x_major_%d.pkl"%(phi2_index), "rb") as fp:
        GBZ_segments = pickle.load(fp)
    print(GBZ_segments)
    for j, seg in enumerate(GBZ_segments):
        ax.plot(np.log(seg[:,1]).imag, np.log(seg[:,2]).imag, seg[:,0].real, 'x')
        for k in range(seg.shape[0]):
            k1 = np.log(seg[k,1]).imag
            k2 = np.log(seg[k,2]).imag
            E = la.eigh(Hamiltonian_matrix(k1, k2, ux, uy, vx))[0]
            # print(E)
            # print(seg[k,0])
            # print(min(np.abs(E-seg[k,0])))
            if(min(np.abs(E-seg[k,0])) > max_diff):
                max_diff = min(np.abs(E-seg[k,0]))
    
    print("max:",max_diff)

    # plt.show()

In [41]:
test_Hermitian(21)

0.4313725490196079
236
Numerical
0
[ 0.57002777+3.70550551e-11j  0.60228433+3.10207872e-11j
  0.63105251+1.84171966e-11j  0.65622051+5.22134760e-12j
  0.67776316-4.63382641e-12j  0.69573203-1.02434463e-11j
  0.71024027+3.35418171e-10j  0.72144503+2.07113789e-11j
  0.72953031+2.57440806e-10j  0.73469202+2.46823629e-10j
  0.73712624+2.45341788e-10j  0.7370209 +2.46613415e-10j
  0.7345504 +2.46735946e-10j  0.72987266+2.44169718e-10j
  0.72312806+2.39377546e-10j  0.71443942+2.33833300e-10j
  0.70391289+2.29028016e-10j  0.69163917+2.26007188e-10j
  0.67769495+2.25327436e-10j  0.66214436+7.05469015e-17j
  0.64504036-2.31568487e-10j  0.62642604-2.38385299e-10j
  0.6063357 -2.47547981e-10j  0.58479589-2.59030701e-10j
  0.56182628-2.72896433e-10j  0.53744032-2.89309483e-10j
  0.51164595-3.08538599e-10j  0.48444606-2.48706498e-10j
  0.455839  +3.57053639e-10j  0.42581894+8.67427324e-12j
  0.39437626+1.16517443e-11j  0.36149791+1.54661093e-11j
  0.32716779+2.03488789e-11j  0.29136719+2.66060355e-

In [30]:
test_Hermitian_y_major(41)

0.823529411764706
Theoretical
Numerical


In [33]:
sweep_x_major()

[]
max: 0.0
[]
max: 0.0
[]
max: 0.0
[]
max: 0.0
[]
max: 0.0


In [34]:
# for j in range(40, 47):
#     test_Hermitian(j)
test_BZ_x_major(41)

NameError: name 'test_BZ_x_major' is not defined

In [33]:
test_Hermitian_y_major(43)

0.8627450980392156
Theoretical


FileNotFoundError: [Errno 2] No such file or directory: 'data/GBZ_data_3/y_major_43.pkl'

In [10]:
import BerryPy.TightBinding as tb
from scipy import linalg as la

PHCv2.4.86 released 2022-05-30 works!


In [11]:
def ssh_2d_Hamiltonian(ux1, ux2, uy1, uy2, vx1, vx2):
    a = 0
    b = 1
    intra_cell = [
        [b, a, ux1],
        [a, b, ux2]
    ]
    inter_cell = [
        [a, b, vx1, (1, 0)],
        [b, a, vx2, (-1, 0)],
        [a, a, uy1, (0, 1)],
        [a, a, uy2, (0, -1)],
        [b, b, vx1, (0, 1)],
        [b, b, vx2, (0, -1)]
    ]

    return tb.TightBindingModel(2, 2, [[1,0],[0,1]],
                                intra_cell, inter_cell)

In [2]:
model = ssh_2d_Hamiltonian(ux1, ux2, uy1, uy2, vx1, vx2)
Nx = 50
Ny = 50
model_x = model.get_supercell(
    [(j,0) for j in range(Nx)],
    np.array([
        [Nx, 0],
        [0, 1]
    ])
)
model_2d = model_x.get_supercell(
    [(0, j) for j in range(Ny)],
    np.array([
        [1,0],
        [0,Ny]
    ])
)
H_dense = model_2d.get_bulk_Hamiltonian_dense((None, None))
eigv, eigvec = la.eig(H_dense)
data = {
    "eigv":eigv,
    "eigvec":eigvec
}
sio.savemat("data/OBC_numerical_1.mat", data)


NameError: name 'ssh_2d_Hamiltonian' is not defined

In [14]:
data = sio.loadmat("data/OBC_numerical_2.mat")

In [15]:
eigv = data['eigv']

In [54]:
# plt.title("OBC, red: x major, blue: y major")
for j in range(50):
# for j in [0, 10, 20, 30]:
    if(os.path.exists("data/GBZ_data_2/x_major_%d.pkl"%(j))):
        with open("data/GBZ_data_2/x_major_%d.pkl"%(j), "rb") as fp:
            GBZ_segments = pickle.load(fp)
        plt.figure(1)
        plt.title("x major")
        plt.axis([-10,10,-3,3])
        for seg in GBZ_segments:
            plt.plot(seg[:,0].real, seg[:,0].imag, 'r.')
    if(os.path.exists("data/GBZ_data_2/y_major_%d.pkl"%(j))):
        with open("data/GBZ_data_2/y_major_%d.pkl"%(j), "rb") as fp:
            GBZ_segments = pickle.load(fp)
        plt.figure(2)
        plt.axis([-10,10,-3,3])
        plt.title("y major")
        for seg in GBZ_segments:
            plt.plot(seg[:,0].real, seg[:,0].imag, 'b.')
    # plt.pause(0.1)
# plt.show()

In [55]:
plt.figure(3)
plt.plot(eigv.real, eigv.imag, 'k.')
plt.axis([-10,10,-3,3])
plt.title("OBC numerical")
# plt.title("Black: OBC numerical, Red: x major GBZ, Blue: y major GBZ")
plt.show()

In [12]:
model = ssh_2d_Hamiltonian(ux1, ux2, uy1, uy2, vx1, vx2)
model_clinic = model.get_supercell(
    [(0,0)],
    [[1, 0],
     [1, 1]]
)
Nx = 30
Ny = 200
model_x = model_clinic.get_supercell(
    [(j,0) for j in range(Nx)],
    np.array([
        [Nx, 0],
        [0, 1]
    ])
)
model_2d = model_x.get_supercell(
    [(0, j) for j in range(Ny)],
    np.array([
        [1,0],
        [0,Ny]
    ])
)
H_dense = model_2d.get_bulk_Hamiltonian_dense((None, None))
eigv, eigvec = la.eig(H_dense)
data = {
    "eigv":eigv,
    "eigvec":eigvec
}
sio.savemat("data/OBC_numerical_2.mat", data)

In [46]:
data = sio.loadmat("data/OBC_numerical_2.mat")
eigv = data['eigv'].flatten()

In [36]:
curr_char_poly = model_clinic.get_characteristic_polynomial_ctype()

/home/wangchenyang/.local/lib/python3.9/site-packages/BerryPy/charpoly_berkowitz.py:61: SymPyDeprecationWarning: 

non-Expr objects in a Matrix is deprecated. Matrix represents
a mathematical matrix. To represent a container of non-numeric
entities, Use a list of lists, TableForm, NumPy array, or some
other data structure instead.

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-non-expr-in-matrix
for details.

This has been deprecated since SymPy version 1.9. It
will be removed in a future version of SymPy.

  toeplitz = sp.SparseMatrix(n+1,n,Elements)


In [39]:
print(curr_char_poly.f_num.to_str(["x", "y", "z"]))

+(+0.000000000000000000e+00-3.000000000000000000e+00*i)*y*(z**3)+(+0.000000000000000000e+00+4.000000000000000000e+00*i)*(y**2)+(-8.000000000000000000e+00+8.000000000000000000e+00*i)*(y**2)*(z**2)+(+0.000000000000000000e+00+4.000000000000000000e+00*i)*(y**2)*(z**4)+(+0.000000000000000000e+00-3.000000000000000000e+00*i)*(y**3)*z+(-4.000000000000000000e+00-1.000000000000000000e+00*i)*x*(y**2)*z+(-4.000000000000000000e+00-1.000000000000000000e+00*i)*x*(y**2)*(z**3)+(+1.000000000000000000e+00+0.000000000000000000e+00*i)*(x**2)*(y**2)*(z**2);


In [40]:
num_coeff, num_orders = curr_char_poly.f_num.batch_get_data()

In [43]:
denom_coeff, denom_orders = curr_char_poly.f_denom.batch_get_data()

In [44]:
print(denom_coeff, denom_orders)

[1.+0.j] [[0 2 2]]


In [45]:
char_poly_orders = num_orders.copy()
for j in range(char_poly_orders.shape[0]):
    for k in range(3):
        char_poly_orders[j, k] -= denom_orders[0, k]

In [46]:
print(num_orders)
print(denom_orders)
print(char_poly_orders)

[[0 1 3]
 [0 2 0]
 [0 2 2]
 [0 2 4]
 [0 3 1]
 [1 2 1]
 [1 2 3]
 [2 2 2]]
[[0 2 2]]
[[ 0 -1  1]
 [ 0  0 -2]
 [ 0  0  0]
 [ 0  0  2]
 [ 0  1 -1]
 [ 1  0 -1]
 [ 1  0  1]
 [ 2  0  0]]


In [47]:
data = {"coeffs":num_coeff, "orders":char_poly_orders}
sio.savemat("data/GBZ_char_poly_2.mat", data)

In [35]:
plt.plot(eigv.real, eigv.imag, '.')
plt.show()

In [49]:
import c_solver_2D as csolver

In [54]:
data_id = 2
char_poly_data = sio.loadmat("data/GBZ_char_poly_%d.mat"%(data_id))
coeffs = csolver.CScalarVec(char_poly_data['coeffs'].flatten())
orders = csolver.CIntVec(char_poly_data['orders'].flatten())

In [52]:
print(coeffs)
print(num_coeff)

aGBZ_2D_solver_ScalarVec[(0,-3), (0,4), (-8,8), (0,4), (0,-3), (-4,-1), (-4,-1), (1,0)]
[ 0.-3.j  0.+4.j -8.+8.j  0.+4.j  0.-3.j -4.-1.j -4.-1.j  1.+0.j]


In [55]:
print(orders)
print(char_poly_orders)

aGBZ_2D_solver_IntVec[0, -1, 1, 0, 0, -2, 0, 0, 0, 0, 0, 2, 0, 1, -1, 1, 0, -1, 1, 0, 1, 2, 0, 0]
[[ 0 -1  1]
 [ 0  0 -2]
 [ 0  0  0]
 [ 0  0  2]
 [ 0  1 -1]
 [ 1  0 -1]
 [ 1  0  1]
 [ 2  0  0]]
